# NT 2.5B vs NT-v2 50M on the genome-wide splice-site dataset

On the small `splice_sites_all` dataset, the frozen NT 2.5B beat the frozen NT-v2 50M by +0.014 macro-F1, with a 95% interval of [+0.006, +0.022] (`results/compare_models/summary.md`). This notebook runs the same two-model comparison on the much bigger genome dataset (`scripts/16_genome_dataset.py`): the 120,000-window class-balanced training file and **every** test window on chr20/21 (18,653 windows from 660 genes). Both models embed exactly the same rows, and the same small head is trained on each.

What happens, in order:

1. Load the two parquet files from the private Kaggle dataset and check them: file hashes, the chromosome split, and that the labels carry the right motifs.
2. For each model (50M first, then 2.5B): find the token that covers the centre of every window, run the frozen model once over every window, and keep two vectors per window (the centre token and the token before it, joined together). The vectors are saved to `/kaggle/working` every 2,048 windows.
3. Train the same head on each model's vectors. 10% of the training windows are set aside for validation, which picks the best epoch.
4. Score each model on the test windows: macro-F1, accuracy, per-class precision and recall, the confusion matrix, and recall on the GT and AG decoy negatives. Every score gets a 95% interval made by **resampling genes**, and the 2.5B-minus-50M difference gets its own paired interval.
5. Write everything into one zip: `/kaggle/working/genome_nt2500m.zip`.

The code is copied from `kaggle/compare_models.ipynb` (embedding, head, scoring, figures), which ran on Kaggle without problems. The new parts are the genome data, the chunked checkpoints, and the gene-level intervals.

**Kaggle settings:** Accelerator = **GPU T4 x2** (only the first GPU is used) or **GPU P100**. Internet = **On** (the models are downloaded from Hugging Face). Input: **Add Input → Your Datasets →** the private dataset with `genome_train.parquet` and `genome_test.parquet`.
First run with `SMOKE = True` (a few minutes) to see that everything is found and runs. Then set `SMOKE = False` and use **Save Version → Save & Run All (Commit)**, so the run continues after you close the browser tab.

**Expected time on a T4:** about 2 hours. This is an estimate, not a measurement: in the `compare_models` run on a T4, NT 2.5B embedded about 29 windows per second and NT-v2 50M about 660. That puts this run's 138,653 windows at about 80 minutes for the 2.5B and 4 minutes for the 50M, plus the 10 GB download and roughly 15 minutes of head training on the CPU. A P100 has no tensor cores, so expect it to be slower, perhaps 3 to 5 hours (a guess). Both fit in Kaggle's 12-hour limit.

**If the run stops:** in the same session, just run all cells again: finished chunks are kept, and only the missing ones are embedded. In a new session, attach the stopped run's output as an input (**Add Input → Your Work / Notebook Output**). Its finished chunks are then copied in automatically before anything is computed.

**Checked on the laptop (CPU) before upload:** the SMOKE run from top to bottom (NT-v2 50M, 400 + 200 windows); its vectors against the local pipeline's saved genome vectors (cosine 1.000000, 98.9% of the float16 values identical); the centre-token lookup on all 138,653 windows with both models' tokenizers (one placement in every window: token 51); resuming after a simulated crash (only the missing chunks recomputed, vectors identical) and reusing a finished checkpoint; the two-model part (difference, intervals, chart, summary, zip) with NT-v2 100M standing in for the 2.5B.

**Not checked, because the laptop has no GPU:** anything on a real GPU, the NT 2.5B weights on this dataset (they did run in `compare_models` on a T4), the P100 PyTorch fix in the first cell, copying a checkpoint from an attached earlier output, and memory use at full size (estimated: about 4 GB of RAM to train the 2.5B head, under 7 GB of GPU memory to embed).

In [ ]:
# Runs before anything imports torch or transformers.
#  - One GPU only: the notebook sees just the first GPU (the "T4 x2" accelerator has two).
#  - Kaggle only: the transformers pin of dna-venv (scripts/setup_local_env.sh).
#  - P100: Kaggle's newest PyTorch builds may no longer contain code for the P100 (compute capability 6.0).
#    A tiny calculation is tried on the GPU in a separate process. If it fails and FIX_OLD_GPU is True,
#    a PyTorch build that still supports the P100 is installed before torch is imported here.
import os
import shutil
import subprocess
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
ON_KAGGLE = os.path.isdir("/kaggle/working")
FIX_OLD_GPU = True
OLD_GPU_TORCH = ["torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cu126"]   # still built for sm_60 (P100)

GPU_PROBE = ("import torch; assert torch.cuda.is_available(), 'torch sees no GPU'; "
             "x = torch.randn(256, 256, device='cuda', dtype=torch.float16); y = (x @ x).float().sum().item(); "
             "torch.cuda.synchronize(); print('ok', torch.__version__, torch.cuda.get_device_name(0), "
             "torch.cuda.get_device_capability(0))")


def gpu_probe():
    r = subprocess.run([sys.executable, "-c", GPU_PROBE], capture_output=True, text=True)
    lines = (r.stdout + r.stderr).strip().splitlines()
    return r.returncode == 0, (lines[-1] if lines else "")


if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0"], check=True)
    if shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0:
        ok, msg = gpu_probe()
        print("GPU test:", "passed" if ok else "FAILED", "|", msg)
        if not ok and FIX_OLD_GPU:
            print("installing a PyTorch build that supports this GPU:", " ".join(OLD_GPU_TORCH))
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *OLD_GPU_TORCH], check=True)
            ok, msg = gpu_probe()
            print("GPU test after the fix:", "passed" if ok else "FAILED", "|", msg)
            if ok and "torch" in sys.modules:
                raise SystemExit("PyTorch was replaced: restart the kernel (Run -> Restart & clear outputs) "
                                 "and run all cells again.")
        if not ok:
            raise SystemExit("PyTorch cannot run on this GPU. Choose Accelerator = GPU T4 x2 and run again.")
    else:
        print("no GPU attached (Settings -> Accelerator). The full run on a CPU would take days.")

## Settings

Everything you might want to change is in the next cell. `SMOKE = True` is a quick test: 400 training windows, 200 test windows, and only the small model. Its numbers mean nothing; it only shows that the data are found and that the notebook runs from top to bottom. A SMOKE run writes into its own folder and never mixes with the real run.

The small model runs first, so it is finished and saved before the big one starts.

In [ ]:
SMOKE = True   # True = 400 train / 200 test windows, NT-v2 50M only (does it run?). False = the real run.

# The two models, in the order they run. Both frozen, inference only. The same pinned
# revisions as kaggle/compare_models.ipynb.
#   weights = "fp32": weights stay full precision on the GPU and the arithmetic runs in half precision
#             (fp16 autocast). "fp16": the weights themselves are half precision. Only for the 2.5B
#             model: its fp32 weights are 10 GB and would nearly fill a 16 GB GPU.
#   download_gb = size of the weight files, used to check there is disk space before downloading
MODELS = [
    {"key": "nt_v2_50m", "name": "NT-v2 50M",
     "repo": "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species",
     "revision": "81b29e5786726d891dbf929404ef20adca5b36f1",
     "weights": "fp32", "download_gb": 0.3},
    {"key": "nt_2500m", "name": "NT 2.5B",
     "repo": "InstaDeepAI/nucleotide-transformer-2.5b-multi-species",
     "revision": "b746b125aacd1b0970c05a32fd71ba726754542e",
     "weights": "fp16", "download_gb": 10.2},
]
REFERENCE, CHALLENGER = "nt_v2_50m", "nt_2500m"   # the difference reported is CHALLENGER minus REFERENCE
SMOKE_MODELS = ["nt_v2_50m"]

# The data: the private Kaggle dataset made by scripts/19_kaggle_genome_package.py.
DATA_FILES = {"train": "genome_train.parquet", "test": "genome_test.parquet"}
LOCAL_DATA_DIRS = ["../data/genome/kaggle_upload/genome-splice-sites",     # run from kaggle/ on the laptop
                   "data/genome/kaggle_upload/genome-splice-sites"]
TRAIN_CHROMS = {f"chr{i}" for i in range(1, 23)} - {"chr20", "chr21"} | {"chrX", "chrY"}   # all but chr20/21
TEST_CHROMS = {"chr20", "chr21"}

SEED = 42                # sampling seed (phase1_common.SEED): the SMOKE sample and the bootstrap redraws
HEAD_SEED = 0            # 03_train_head.py --seed: validation split and head initialisation
N_TRAIN, N_TEST = (400, 200) if SMOKE else (-1, -1)     # -1 = every window
VAL_FRAC = 0.10          # 03_train_head.py --val-frac
CENTER_BASE = 300        # 02b_embed_center.py / 12_repool_center.py

EMBED_BATCH = 64         # windows per forward pass; halved automatically if the GPU runs out of memory
MIN_BATCH = 4            # give up on a model if even this does not fit
CHUNK = 2048             # windows per checkpoint (a multiple of EMBED_BATCH)
FP16_MIN_COSINE = 0.99   # first batch: fp16 vectors must point the same way as fp32 vectors
FREE_MODEL_CACHE = True  # on Kaggle only: delete each model's download once its embeddings are saved
ROW_CHUNK = 20_000       # rows per step when gathering and standardising vectors (keeps memory low)

# Head recipe = 03_train_head.py defaults, as in compare_models.ipynb.
HEAD_LR, HEAD_WEIGHT_DECAY, HEAD_BATCH, HEAD_MAX_EPOCHS, HEAD_PATIENCE = 1e-3, 0.0, 128, 300, 20

N_BOOT = 2000            # bootstrap redraws of the test GENES, for the 95% intervals
ZIP_EMBEDDINGS = False   # True = also put the saved vectors in the zip (about 1.7 GB), so other heads can be tried later

LABEL_NAMES = {0: "none", 1: "acceptor", 2: "donor"}   # data/label_info.json; re-checked below
MOTIF_OFFSETS = {"donor_GT_plus": 301, "donor_AC_minus": 298,
                 "acceptor_AG_plus": 298, "acceptor_CT_minus": 301}
NEG_GROUPS = ["decoy_GT", "decoy_AG", "exon_interior", "intron_interior"]

# NT-v2 50M on the same 120,000 train / 18,653 test windows on the laptop (CPU), mean of 3 head seeds,
# results/learning_curve/summary.md. The 50M here should land close to these.
LAPTOP_50M = {"macro_f1": 0.957, "decoy_GT": 0.891, "decoy_AG": 0.827}

In [ ]:
import collections, copy, gc, hashlib, json, platform, shutil, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from torch import nn

T_START = time.time()
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")   # one GPU, no DataParallel
ON_GPU = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if ON_GPU else None
GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if ON_GPU else None
GPU_CC = ".".join(map(str, torch.cuda.get_device_capability(0))) if ON_GPU else None

BASE_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("GENOME_NB_OUT", Path.cwd()))
OUT_DIR = BASE_DIR / ("genome_nt2500m_smoke" if SMOKE else "genome_nt2500m")   # SMOKE never mixes with the real run
EMB_DIR, FIG_DIR, PRED_DIR, HEAD_DIR = (OUT_DIR / d for d in ("embeddings", "figures", "predictions", "heads"))
for d in (EMB_DIR, FIG_DIR, PRED_DIR, HEAD_DIR):
    d.mkdir(parents=True, exist_ok=True)
RESULTS_JSON, SUMMARY_MD = OUT_DIR / "results.json", OUT_DIR / "summary.md"
ZIP_PATH = BASE_DIR / f"{OUT_DIR.name}.zip"

RUN_MODELS = [m for m in MODELS if not SMOKE or m["key"] in SMOKE_MODELS]
SPECS = {m["key"]: m for m in MODELS}
assert len(SPECS) == len(MODELS) and REFERENCE in SPECS and CHALLENGER in SPECS
assert CHUNK % EMBED_BATCH == 0

print(f"python {platform.python_version()}  torch {torch.__version__}  "
      f"transformers {transformers.__version__}  sklearn {sklearn.__version__}")
print(f"device: {DEVICE}" + (f" ({GPU_NAME}, compute capability {GPU_CC}, {GPU_TOTAL_GB:.1f} GB)" if ON_GPU else "")
      + f"   half precision (fp16): {ON_GPU}")
print(f"SMOKE = {SMOKE}: models {[m['name'] for m in RUN_MODELS]}")
print(f"output folder: {OUT_DIR}")
assert transformers.__version__ == "4.49.0", "restart the kernel so the pinned transformers is the one imported"
if not SMOKE and not ON_GPU:
    print("WARNING: the real run on a CPU would take days. On Kaggle: Settings -> Accelerator -> GPU T4 x2.")

## Data: the genome dataset, chromosome split, every window

Two files from the private Kaggle dataset. The training file is the 120,000-window class-balanced default file (40,000 donors, 40,000 acceptors, and 40,000 negatives), from every chromosome except 20 and 21 (chr1–19, chr22, X, Y). The test file holds every window on chromosomes 20 and 21. So the test never contains a piece of DNA the head has seen.

The negatives are of four kinds: **GT decoys** (a GT at the donor position that is not a splice site in any GENCODE transcript), **AG decoys** (the same for acceptors), exon interiors, and intron interiors. The decoys are the hard ones: the motif is there, but the site is not real.

The cell checks the file hashes against `manifest.json`, the chromosome split, and that no test sequence appears in train. It also re-checks what the label numbers mean, the way `01_get_data.py` found it: donors carry `GT` just after the centre, acceptors carry `AG` just before it. One tenth of the training windows is set aside as validation; the test windows are used only once, at the end.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def find_data_dir():
    # The folder holding both parquet files: $GENOME_DATA_DIR, then anything attached under
    # /kaggle/input, then the laptop's package folder.
    cands = [Path(os.environ["GENOME_DATA_DIR"])] if os.environ.get("GENOME_DATA_DIR") else []
    if ON_KAGGLE:
        cands += sorted({p.parent for p in Path("/kaggle/input").rglob(DATA_FILES["train"])})
    cands += [Path(p) for p in LOCAL_DATA_DIRS]
    for d in cands:
        if all((d / f).is_file() for f in DATA_FILES.values()):
            return d.resolve()
    raise FileNotFoundError(f"{' and '.join(DATA_FILES.values())} not found. On Kaggle: Add Input -> "
                            f"Your Datasets -> the private genome dataset. Looked in: {[str(c) for c in cands]}")


def stratified_subsample(labels, n, seed=SEED):
    # phase1_common.py stratified_subsample(), unchanged. n < 0 means "every row".
    labels = np.asarray(labels)
    total = len(labels)
    if n is None or n < 0 or n >= total:
        return np.arange(total)
    from sklearn.model_selection import train_test_split

    idx, _ = train_test_split(np.arange(total), train_size=n, stratify=labels, random_state=seed)
    return np.sort(idx)


def indices_digest(idx):
    # 02_embed.py indices_digest(), unchanged: a short fingerprint of which rows were used
    return hashlib.sha1(np.ascontiguousarray(idx, dtype=np.int64).tobytes()).hexdigest()[:16]


DATA_DIR = find_data_dir()
DATA_SHA = {s: sha256(DATA_DIR / f) for s, f in DATA_FILES.items()}
print(f"data folder: {DATA_DIR}")
MANIFEST = json.loads((DATA_DIR / "manifest.json").read_text()) if (DATA_DIR / "manifest.json").exists() else None
if MANIFEST:
    for s, f in DATA_FILES.items():
        assert DATA_SHA[s] == MANIFEST["files"][f]["sha256"], \
            f"{f}: sha256 differs from manifest.json (the file changed or was damaged in the upload)"
    print("file hashes match manifest.json")
else:
    print("WARNING: no manifest.json next to the data files, so their hashes cannot be checked")

full = {s: pd.read_parquet(DATA_DIR / f) for s, f in DATA_FILES.items()}
for s, f in DATA_FILES.items():
    if MANIFEST:
        assert len(full[s]) == MANIFEST["files"][f]["rows"], f"{f}: row count differs from manifest.json"
    assert (full[s]["sequence"].str.len() == 600).all(), f"{f}: a window is not 600 nt long"

# Chromosome split: never a train window from chr20/21, never a test window from anywhere else
chroms = {s: set(full[s]["chrom"].astype(str)) for s in full}
assert chroms["train"] <= TRAIN_CHROMS, f"unexpected train chromosomes: {sorted(chroms['train'] - TRAIN_CHROMS)}"
assert chroms["test"] <= TEST_CHROMS, f"unexpected test chromosomes: {sorted(chroms['test'] - TEST_CHROMS)}"
assert not set(full["test"]["sequence"]) & set(full["train"]["sequence"]), "a test window also appears in train"
print(f"train: {len(full['train']):,} rows on {len(chroms['train'])} chromosomes (all but chr20/21)   "
      f"test: {len(full['test']):,} rows on {', '.join(sorted(chroms['test']))}")

idx = {s: stratified_subsample(full[s]["label"], n, SEED) for s, n in (("train", N_TRAIN), ("test", N_TEST))}
seqs = {s: full[s]["sequence"].iloc[idx[s]].tolist() for s in idx}
y = {s: full[s]["label"].to_numpy()[idx[s]].astype(np.int64) for s in idx}
TEST_GENE = full["test"]["gene_id"].astype(str).to_numpy(dtype=str)[idx["test"]]
TEST_GROUP = full["test"]["group"].astype(str).to_numpy(dtype=str)[idx["test"]]
GENE_CODE, GENES = pd.factorize(TEST_GENE)
CLASSES = sorted(LABEL_NAMES)
CLASS_NAMES = [LABEL_NAMES[c] for c in CLASSES]
assert all(set(y[s].tolist()) <= set(CLASSES) for s in y)
assert set(TEST_GROUP[y["test"] == 0]) <= set(NEG_GROUPS)
if not SMOKE:
    assert all(len(idx[s]) == len(full[s]) for s in idx), "the real run must use every window"
ROWS_KEY = {s: {"n": int(len(idx[s])), "sha1": indices_digest(idx[s]), "file_sha256": DATA_SHA[s]} for s in idx}

# 10% validation rows carved from train: the split 03_train_head.py makes
# (train_test_split(test_size=0.10, stratify=y, random_state=seed)). Both models use the same rows.
from sklearn.model_selection import train_test_split
fit_rows, val_rows = train_test_split(np.arange(len(y["train"])), test_size=VAL_FRAC,
                                      stratify=y["train"], random_state=HEAD_SEED)
for s in idx:
    print(f"{s:>5}: {len(idx[s]):,} rows, per class { {LABEL_NAMES[c]: int((y[s] == c).sum()) for c in CLASSES} }")
print(f"train -> {len(fit_rows):,} to fit the head + {len(val_rows):,} validation")
print(f"test: {len(GENES)} genes; kinds of window "
      f"{ {g: int((TEST_GROUP == g).sum()) for g in ['donor', 'acceptor'] + NEG_GROUPS} }")

# Label names re-checked the way 01_get_data.py found them: donors carry GT at offset 301
# (or AC at 298 on the other strand), acceptors AG at 298 (or CT at 301).
def has(seq_list, a, ka, b, kb):
    return float(np.mean([s[ka:ka + 2] == a or s[kb:kb + 2] == b for s in seq_list]))


o = MOTIF_OFFSETS
for c in CLASSES:
    rows = [s for s, lab in zip(seqs["train"], y["train"]) if lab == c]
    d = has(rows, "GT", o["donor_GT_plus"], "AC", o["donor_AC_minus"])
    a = has(rows, "AG", o["acceptor_AG_plus"], "CT", o["acceptor_CT_minus"])
    print(f"label {c} ({LABEL_NAMES[c]:>8}): donor motif {d:.1%}   acceptor motif {a:.1%}")
    if LABEL_NAMES[c] == "donor":
        assert d > 0.9 and a < 0.3, "label 2 does not look like donor"
    if LABEL_NAMES[c] == "acceptor":
        assert a > 0.9 and d < 0.3, "label 1 does not look like acceptor"

## Tokens: which piece of the model's input is the centre?

A model does not read DNA letter by letter. Its **tokenizer** first cuts the 600 letters into pieces called tokens, and the model gives back one vector per token. The label depends on the letters at the centre of the window, so we want the vector of the token that covers letter 300, plus the token just before it.

Both Nucleotide Transformers cut the DNA into 6-letter pieces, so the centre is usually the same token in every window, but one `N` would shift everything. So nothing is hard-coded: for **every window**, the code below maps each token back to the letters it covers, checks that the tokens put back together spell the original sequence exactly, and only then looks up the centre token and the one before it. If any check fails it stops with an error, because a wrong position would not crash: it would quietly read the wrong place and give a worse score. (Copied unchanged from `compare_models.ipynb`.)

In [ ]:
from transformers import AutoConfig, AutoTokenizer


def load_tokenizer(spec):
    # The tokenizer and the model's settings file (config). Both are tiny downloads.
    tok = AutoTokenizer.from_pretrained(spec["repo"], revision=spec["revision"], trust_remote_code=True)
    config = AutoConfig.from_pretrained(spec["repo"], revision=spec["revision"], trust_remote_code=True)
    special_ids = np.array(sorted(set(tok.all_special_ids)))
    return tok, special_ids, config


def token_limit(tok, config):
    # 02b_embed_center.py: the smallest of the tokenizer's limit, the model's limit and 2048
    limits = [getattr(tok, "model_max_length", None),
              getattr(config, "max_position_embeddings", None), 2048]
    return int(min(x for x in limits if x and x < 1_000_000))


TOKENIZERS = {}


def encode_batch(tok, special_ids, max_len, seq_list, seen):
    # compare_models.ipynb encode_batch(), unchanged.
    # Tokenise a batch and find, per sequence, the input position of the token covering base 300
    # (C) and of the token just before it (P).
    # Token -> base mapping from 02b_embed_center.py spans():
    #   fast tokenizer: the tokenizer reports each token's [start, end) offsets
    #   slow tokenizer (NT): cumulative lengths of the token strings
    # Either way the tokens must tile the sequence and spell it exactly.
    # Special tokens are found by id: NT's slow tokenizer returns a special_tokens_mask that
    # assumes an EOS token it never adds (off by one).
    enc = tok(seq_list, return_tensors="pt", padding=True, truncation=True, max_length=max_len,
              return_offsets_mapping=bool(tok.is_fast))
    ids, am = enc["input_ids"], enc["attention_mask"]
    ids_np, am_np = ids.numpy(), am.numpy().astype(bool)
    c_idx = np.empty(len(seq_list), dtype=np.int64)
    p_idx = np.empty(len(seq_list), dtype=np.int64)
    for i, seq in enumerate(seq_list):
        body = am_np[i] & ~np.isin(ids_np[i], special_ids)
        pos = np.flatnonzero(body)                     # input positions of the real tokens
        pieces = tok.convert_ids_to_tokens(ids_np[i][body].tolist())
        if tok.is_fast:
            off = enc["offset_mapping"][i].numpy()[body]
            start, end = off[:, 0].astype(np.int64), off[:, 1].astype(np.int64)
            assert start[0] == 0 and end[-1] == len(seq) and (start[1:] == end[:-1]).all(), \
                "token offsets do not tile the sequence"
            assert all(seq[a:b] == p for a, b, p in zip(start, end, pieces)), \
                "tokens do not spell the bases at their offsets"
        else:
            lens = np.array([len(p) for p in pieces])
            assert int(lens.sum()) == len(seq), \
                f"reconstructed length {int(lens.sum())} != sequence length {len(seq)}"
            assert "".join(pieces) == seq, "reconstructed tokens do not spell the sequence"
            end = np.cumsum(lens)
            start = end - lens
        c = np.flatnonzero((start <= CENTER_BASE) & (end > CENTER_BASE))
        assert len(c) == 1, f"expected exactly one token covering base {CENTER_BASE}"
        c = int(c[0])
        assert c >= 1, "no token before the centre token"
        assert end[c - 1] == start[c], "previous token is not adjacent"
        c_idx[i], p_idx[i] = pos[c], pos[c - 1]
        seen[(int(pos[c]), int(start[c]), int(end[c]) - 1,
              int(pos[c - 1]), int(start[c - 1]), int(end[c - 1]) - 1)] += 1
    return ids, am, torch.from_numpy(c_idx), torch.from_numpy(p_idx)


def tokenise_all(spec):
    # Every window of both splits, tokenised once in batches of EMBED_BATCH, with the centre and
    # previous token found and checked. The model is later run on exactly these tensors.
    tok, special_ids, config = load_tokenizer(spec)
    TOKENIZERS[spec["key"]] = tok
    max_len = token_limit(tok, config)
    seen, t0 = collections.Counter(), time.time()
    batches = {split: [encode_batch(tok, special_ids, max_len, seqs[split][i:i + EMBED_BATCH], seen)
                       for i in range(0, len(seqs[split]), EMBED_BATCH)] for split in ("train", "test")}
    centre = centre_summary(seen)
    centre["tokenizer"] = type(tok).__name__ + (" (fast: offsets)" if tok.is_fast else " (slow: token lengths)")
    centre["max_tokens"] = max_len
    centre["tokens_per_window"] = [int(min(int(b[1].sum(1).min()) for s in batches for b in batches[s])),
                                   int(max(int(b[1].sum(1).max()) for s in batches for b in batches[s]))]
    centre["tokenise_seconds"] = round(time.time() - t0, 1)
    return batches, max_len, config, centre


def centre_summary(seen, top=5):
    # What the lookup found, for results.json and for reading by eye
    total = sum(seen.values())
    rows = [{"rows": n, "share": round(n / total, 4), "centre_input_index": ci, "centre_bases": [cs, ce],
             "previous_input_index": pi, "previous_bases": [ps, pe]}
            for (ci, cs, ce, pi, ps, pe), n in seen.most_common(top)]
    return {"windows_checked": total, "distinct_placements": len(seen), "most_common": rows}

## Running a frozen model, with checkpoints

**Frozen** means the model's own numbers never change: it is put in evaluation mode and run without gradients, so this is plain inference. Each window goes through each model exactly once.

**Half precision (fp16)** means the GPU does the arithmetic with 16-bit numbers instead of 32-bit ones, which is faster and needs less memory. For the 50M model, the first 16 windows are run both ways and compared. The 2.5B model also stores its weights in half precision, because the full-precision copy would nearly fill the GPU; the same setting worked in the `compare_models` run.

**If something goes wrong.** Running out of GPU memory halves the batch size and tries again. Any other failure in half precision gets one retry in full precision. If a model still cannot run, it is skipped, the reason is written to `results.json`, and the notebook carries on.

**Checkpoints.** Each model's vectors go into two files, `embeddings/<model>/train.npy` and `test.npy` (float16, the same rounding the local pipeline applies). After every 2,048 windows the file is flushed to disk and `progress.json` records which chunks are finished. A rerun skips finished chunks, so a crash loses at most one chunk. A checkpoint is used only if it was made from the same rows, the same model revision and the same precision.

In [ ]:
def load_model(spec, half_weights):
    # try_dna_models.py load_model() via compare_models.ipynb: NT-v2 registers AutoModelForMaskedLM
    # but not AutoModel, so both are tried.
    from transformers import AutoModel, AutoModelForMaskedLM

    kwargs = {"revision": spec["revision"], "trust_remote_code": True}
    if half_weights:
        kwargs["torch_dtype"] = torch.float16
    errors = []
    for cls in (AutoModel, AutoModelForMaskedLM):
        try:
            return cls.from_pretrained(spec["repo"], **kwargs), cls.__name__
        except Exception as exc:
            errors.append(f"{cls.__name__}: {type(exc).__name__}: {str(exc)[:300]}")
    raise RuntimeError(" | ".join(errors))


def token_states(model, input_ids, attention_mask):
    # try_dna_models.py get_hidden(): last-layer vector of every token
    out = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state
    if getattr(out, "hidden_states", None):
        return out.hidden_states[-1]
    if isinstance(out, (tuple, list)):
        return out[0]
    raise RuntimeError("could not find per-token vectors in the model output")


def centre_features(model, ids, am, c, p, autocast):
    # center_plus_prev_concat: [centre token ++ previous token], in that order
    # (12_repool_center.py: np.concatenate([cen, prev], axis=1)). No gradients: the model is frozen.
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=autocast):
        h = token_states(model, ids.to(DEVICE), am.to(DEVICE))
        r = torch.arange(h.shape[0], device=h.device)
        return torch.cat([h[r, c.to(DEVICE)], h[r, p.to(DEVICE)]], dim=-1).float().cpu().numpy()


def is_out_of_memory(exc):
    return "out of memory" in str(exc).lower()


def free_gpu():
    gc.collect()
    if ON_GPU:
        torch.cuda.empty_cache()


def model_cache_dir(spec):
    from huggingface_hub.constants import HF_HUB_CACHE
    return Path(HF_HUB_CACHE) / ("models--" + spec["repo"].replace("/", "--"))


def weights_downloaded(spec):
    # True if a weight file of this model is already in the download cache
    cache = model_cache_dir(spec)
    return cache.exists() and any(f.is_file() and not f.name.endswith(".incomplete") and f.stat().st_size > 50e6
                                  for f in cache.rglob("*"))


def free_disk_gb(path):
    path = Path(path)
    while not path.exists():
        path = path.parent
    return shutil.disk_usage(path).free / 1e9


def write_json_atomic(path, obj):
    # phase1_common.py write_json_atomic(): via a temp file, so a crash never leaves half a file
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2)
        fh.write("\n")
    os.replace(tmp, path)


# ---------- checkpoints ----------
SPLITS = ("test", "train")       # the order the windows are embedded in


def store_dir(spec):
    return EMB_DIR / spec["key"]


def run_key(spec):
    # What a checkpoint must match to be reused
    return json.loads(json.dumps({"repo": spec["repo"], "revision": spec["revision"], "rows": ROWS_KEY,
                                  "center_base": CENTER_BASE, "features": "center_plus_prev_concat",
                                  "chunk": CHUNK}))


def n_chunks(split):
    return -(-len(seqs[split]) // CHUNK)


def read_progress(spec, folder=None):
    # The checkpoint's progress record if it matches this run (rows, model, settings), else None
    folder = Path(folder) if folder else store_dir(spec)
    p = folder / "progress.json"
    if not p.exists():
        return None
    prog = json.loads(p.read_text())
    if prog.get("run_key") != run_key(spec):
        print(f"  checkpoint in {folder} is for other rows or settings: not used")
        return None
    for s in SPLITS:
        f = folder / f"{s}.npy"
        if not f.exists() or np.load(f, mmap_mode="r").shape != (len(seqs[s]), prog["feature_dim"]):
            print(f"  checkpoint in {folder}: {s}.npy is missing or has the wrong shape: not used")
            return None
    return prog


def chunks_done(prog):
    return 0 if prog is None else sum(len(prog["chunks_done"][s]) for s in SPLITS)


def is_complete(prog):
    return prog is not None and all(len(set(prog["chunks_done"][s])) == n_chunks(s) for s in SPLITS)


def import_previous_run(spec):
    # Kaggle only: if the output of an earlier, stopped run is attached as an input, copy its
    # checkpoint in when it has more finished chunks than the one here.
    if not ON_KAGGLE:
        return
    here = chunks_done(read_progress(spec))
    for p in sorted(Path("/kaggle/input").rglob(f"{OUT_DIR.name}/embeddings/{spec['key']}/progress.json")):
        prog = read_progress(spec, p.parent)
        if chunks_done(prog) > here:
            print(f"  copying {chunks_done(prog)} finished chunks from {p.parent}")
            store_dir(spec).mkdir(parents=True, exist_ok=True)
            for s in SPLITS:
                shutil.copyfile(p.parent / f"{s}.npy", store_dir(spec) / f"{s}.npy")
            shutil.copyfile(p, store_dir(spec) / "progress.json")
            here = chunks_done(prog)


def open_store(spec, label, feature_dim):
    # The two float16 files, opened for writing. Created empty when there is no usable checkpoint,
    # or when the checkpoint was made with another precision (vectors are never mixed).
    d = store_dir(spec)
    d.mkdir(parents=True, exist_ok=True)
    prog = read_progress(spec)
    if prog is not None and (prog["precision"] != label or prog["feature_dim"] != feature_dim):
        print(f"  checkpoint was made with {prog['precision']}: starting this model again with {label}")
        prog = None
    if prog is None:
        need = sum(len(seqs[s]) for s in SPLITS) * feature_dim * 2 / 1e9
        if free_disk_gb(d) < need + 0.5:
            raise RuntimeError(f"not enough disk for the vectors: {free_disk_gb(d):.1f} GB free, need {need:.1f} GB")
        mms = {s: np.lib.format.open_memmap(d / f"{s}.npy", mode="w+", dtype=np.float16,
                                            shape=(len(seqs[s]), feature_dim)) for s in SPLITS}
        prog = {"run_key": run_key(spec), "precision": label, "feature_dim": feature_dim,
                "chunks_done": {s: [] for s in SPLITS}, "embed_seconds": 0.0, "sessions": 0,
                "peak_gpu_gb": None}
    else:
        mms = {s: np.load(d / f"{s}.npy", mmap_mode="r+") for s in SPLITS}
        print(f"  resuming from the checkpoint: {chunks_done(prog)} of "
              f"{sum(n_chunks(s) for s in SPLITS)} chunks already done")
    prog["sessions"] += 1
    write_json_atomic(d / "progress.json", prog)
    return mms, prog


def embed_once(spec, batches, max_len, centre, half_weights, autocast):
    # One attempt at embedding every window not yet in the checkpoint, with one precision setting.
    label = ("fp16 weights" if half_weights else "fp32 weights") + (" + fp16 autocast" if autocast else ", no autocast")
    need = spec.get("download_gb", 0) + 2
    if not weights_downloaded(spec) and free_disk_gb(model_cache_dir(spec)) < need:
        raise RuntimeError(f"not enough disk for the download: {free_disk_gb(model_cache_dir(spec)):.1f} GB free, "
                           f"need about {need:.1f} GB")
    if ON_GPU:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    model, loaded_via = load_model(spec, half_weights)
    try:
        model.eval()
        for prm in model.parameters():
            prm.requires_grad_(False)
        model.to(DEVICE)
        load_s = time.time() - t0
        n_params = sum(prm.numel() for prm in model.parameters())
        hidden = int(model.config.hidden_size)
        assert token_limit(TOKENIZERS[spec["key"]], model.config) == max_len, \
            "the model's token limit differs from the one used to tokenise"
        print(f"  loaded via {loaded_via} in {load_s:.0f}s ({label}): {n_params / 1e6:.0f}M parameters, "
              f"hidden size {hidden}, head input {2 * hidden}"
              + (f", GPU memory after load {torch.cuda.memory_allocated() / 1e9:.1f} GB" if ON_GPU else ""))
        info = {"precision": label, "loaded_via": loaded_via, "parameters": int(n_params), "hidden_size": hidden,
                "feature_dim": 2 * hidden, "max_tokens": max_len, "load_seconds": round(load_s, 1),
                "fp16_check": None}
        if half_weights:
            info["fp16_check"] = "not run: the weights are fp16, so there is no fp32 result to compare with"
        elif autocast:
            # First 16 windows only: the same windows in full precision and in half precision.
            ids, am, c, p = (t[:16] for t in batches["train"][0])
            f32 = centre_features(model, ids, am, c, p, autocast=False)
            f16 = centre_features(model, ids, am, c, p, autocast=True)
            cos = (f32 * f16).sum(1) / (np.linalg.norm(f32, axis=1) * np.linalg.norm(f16, axis=1))
            info["fp16_check"] = {"windows": int(len(ids)), "min_cosine_vs_fp32": round(float(cos.min()), 6),
                                  "max_abs_diff": round(float(np.abs(f32 - f16).max()), 4)}
            print(f"  fp16 check on the first {len(ids)} windows: cosine with fp32 at least "
                  f"{cos.min():.5f}, largest difference {np.abs(f32 - f16).max():.3f}")
            if not np.isfinite(cos).all() or cos.min() < FP16_MIN_COSINE:
                raise ValueError(f"fp16 output differs from fp32 (min cosine {cos.min():.4f})")

        mms, prog = open_store(spec, label, 2 * hidden)
        done = {s: set(prog["chunks_done"][s]) for s in SPLITS}
        todo = [(s, k) for s in SPLITS for k in range(n_chunks(s)) if k not in done[s]]
        n_left = sum(min(CHUNK, len(seqs[s]) - k * CHUNK) for s, k in todo)
        print(f"  {len(todo)} chunks to embed ({n_left:,} windows)")
        per_chunk, bs = CHUNK // EMBED_BATCH, EMBED_BATCH
        seconds_before, n_done, t_run, next_report = prog["embed_seconds"], 0, time.time(), 0.0
        for s, k in todo:
            out = []
            for ids, am, c, p in batches[s][k * per_chunk:(k + 1) * per_chunk]:
                i = 0
                while i < len(ids):
                    try:
                        feats = centre_features(model, ids[i:i + bs], am[i:i + bs], c[i:i + bs], p[i:i + bs], autocast)
                    except RuntimeError as exc:
                        if not is_out_of_memory(exc) or bs <= MIN_BATCH:
                            raise
                        bs = max(MIN_BATCH, bs // 2)
                        free_gpu()
                        print(f"  GPU out of memory: batch size lowered to {bs}, trying again")
                        continue
                    if not np.isfinite(feats).all():
                        raise ValueError(f"non-finite values in the embeddings ({label})")
                    out.append(feats)
                    i += len(feats)
            # phase1_common.py pair_features(): this variant goes to the head rounded through float16,
            # so the vectors are stored as float16 (the same rounding, half the disk).
            F = np.concatenate(out).astype(np.float16)
            if not np.isfinite(F).all():
                raise ValueError("a value does not fit in float16")
            a = k * CHUNK
            assert a + len(F) == min((k + 1) * CHUNK, len(seqs[s]))
            mms[s][a:a + len(F)] = F
            mms[s].flush()
            prog["chunks_done"][s].append(k)
            prog["embed_seconds"] = round(seconds_before + time.time() - t_run, 1)
            if ON_GPU:
                prog["peak_gpu_gb"] = round(max(prog["peak_gpu_gb"] or 0, torch.cuda.max_memory_allocated() / 1e9), 2)
            write_json_atomic(store_dir(spec) / "progress.json", prog)
            n_done += len(F)
            el = time.time() - t_run
            if el >= next_report or n_done == n_left:
                print(f"  {n_done:7,d}/{n_left:,} windows  {el:6.0f}s elapsed  {n_done / el:6.1f} windows/s  "
                      f"about {(n_left - n_done) * el / n_done / 60:5.1f} min left", flush=True)
                next_report = el + max(30.0, el * 0.2)
        if ON_GPU:
            torch.cuda.synchronize()
        del mms
        n_all = sum(len(seqs[s]) for s in SPLITS)
        info.update({"batch_size_final": bs, "embed_seconds": prog["embed_seconds"],
                     "embed_seconds_this_session": round(time.time() - t_run, 1),
                     "windows_this_session": int(n_done), "sessions": prog["sessions"],
                     "embed_ms_per_window": round(1000 * prog["embed_seconds"] / n_all, 2),
                     "peak_gpu_gb": prog["peak_gpu_gb"],
                     "peak_gpu_reserved_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2) if ON_GPU else None})
        prog.update({"embedding": info, "centre_token": centre})
        write_json_atomic(store_dir(spec) / "progress.json", prog)
        return info
    finally:
        del model
        free_gpu()


def embed_model(spec):
    # Vectors for one model: from a complete checkpoint if there is one, otherwise computed (or
    # finished) and saved. Returns (embedding info, centre-token summary).
    import_previous_run(spec)
    prog = read_progress(spec)
    if is_complete(prog):
        print(f"  every window already embedded (checkpoint {store_dir(spec)}): not recomputed")
        return dict(prog.get("embedding", {}), from_checkpoint=True), prog.get("centre_token", {})

    # Tokenise and check the centre token in every window BEFORE the model is downloaded:
    # a tokenizer problem should cost seconds, not a 10 GB download.
    batches, max_len, config, centre = tokenise_all(spec)
    print(f"  centre token found and checked in all {centre['windows_checked']:,} windows "
          f"({centre['tokenise_seconds']:.0f}s, {centre['tokenizer']}, {centre['tokens_per_window'][0]}-"
          f"{centre['tokens_per_window'][1]} tokens per window); {centre['distinct_placements']} placement(s):")
    for r in centre["most_common"][:3]:
        print(f"    {r['rows']:7,d} windows: centre token = input {r['centre_input_index']} (bases "
              f"{r['centre_bases'][0]}-{r['centre_bases'][1]}), previous = input {r['previous_input_index']} "
              f"(bases {r['previous_bases'][0]}-{r['previous_bases'][1]})")

    half = ON_GPU and spec["weights"] == "fp16"
    attempts = [(half, ON_GPU)]                 # (half-precision weights, fp16 autocast)
    if ON_GPU:
        attempts.append((False, False))         # one retry in full precision
    errors = []
    for half_weights, autocast in attempts:
        try:
            info = embed_once(spec, batches, max_len, centre, half_weights, autocast)
            break
        except Exception as exc:
            free_gpu()
            errors.append(f"{type(exc).__name__}: {str(exc)[:400]}")
            print(f"  attempt failed: {errors[-1]}")
    else:
        raise RuntimeError(" || ".join(errors))
    del batches
    prog = read_progress(spec)
    assert is_complete(prog), "embedding finished but the checkpoint is not complete"
    info.update({"failed_attempts": errors, "from_checkpoint": False, "tokenise_seconds": centre["tokenise_seconds"]})
    prog["embedding"] = info
    write_json_atomic(store_dir(spec) / "progress.json", prog)
    print(f"  saved {store_dir(spec)}: {prog['feature_dim']} values per window; embedding took "
          f"{info['embed_seconds'] / 60:.1f} min" + (f", peak GPU memory {info['peak_gpu_gb']:.1f} GB" if ON_GPU else ""))
    if ON_KAGGLE and FREE_MODEL_CACHE:
        shutil.rmtree(model_cache_dir(spec), ignore_errors=True)     # the download is no longer needed
    return info, centre


def features(spec):
    # The saved vectors (float16, read from disk as needed)
    return {s: np.load(store_dir(spec) / f"{s}.npy", mmap_mode="r") for s in SPLITS}

## The head, and how it is scored

The head is the only part that learns. It is the same small network for both models: the vector goes into 256 units, a ReLU, dropout of 0.2, and out to 3 scores (none / acceptor / donor). Only its input size changes: 1,024 values for the 50M model, 5,120 for the 2.5B.

Training follows `03_train_head.py`, as in `compare_models.ipynb`: each input column is standardised (mean 0, spread 1, measured on the training part only); the optimiser is Adam; after every pass the head is scored on the validation part; training stops once validation has not improved for 20 passes; and the weights from the best validation pass are kept. The test windows play no part in any of this. With 108,000 training rows the standardising is done in steps of 20,000 rows to keep memory low; the result is the same up to rounding.

**The numbers reported.** Macro-F1 (the plain average of the three classes' F1) is the headline. Also reported: accuracy, per-class precision (of the windows *called* donor, how many are donors) and recall (of the real donors, how many were found), the confusion matrix, and for each kind of negative the share the head correctly calls none. That share is the "recall on the GT and AG decoys".

**95% intervals, resampling genes.** Windows from the same gene share sequence and context, so they are not independent. Resampling single windows would make the intervals too narrow. Instead, each of the 2,000 redraws picks the test **genes** with replacement (660 of them, as many as there are) and takes all windows of every gene drawn. Both models are scored on the same redraws, so the 2.5B-minus-50M difference has a proper paired interval: if it does not include 0, the difference is unlikely to be luck. The window-level interval is also stored in `results.json`, to show how much wider the gene-level one is.

In [ ]:
from sklearn.preprocessing import StandardScaler

to_idx = {c: i for i, c in enumerate(CLASSES)}
yi = {s: np.array([to_idx[v] for v in y[s]]) for s in y}
NONE_I = to_idx[0]


def make_head(in_dim):
    # 03_train_head.py: input -> 256 -> ReLU -> Dropout(0.2) -> classes
    return nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, len(CLASSES)))


def train_mlp(X_fit, y_fit, X_val, y_val):
    # 03_train_head.py train_mlp() via compare_models.ipynb: Adam, early stopping on validation loss,
    # best weights restored. On CPU. (torch.from_numpy instead of torch.tensor: same numbers, no copy.)
    torch.manual_seed(HEAD_SEED)
    gen = torch.Generator().manual_seed(HEAD_SEED)
    Xf, yf = torch.from_numpy(X_fit), torch.tensor(y_fit, dtype=torch.long)
    Xv, yv = torch.from_numpy(X_val), torch.tensor(y_val, dtype=torch.long)
    model = make_head(Xf.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    best = {"val_loss": float("inf"), "epoch": 0, "state": None}
    history, since_best, t0 = [], 0, time.time()
    for epoch in range(1, HEAD_MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(Xf), generator=gen)
        total = 0.0
        for i in range(0, len(Xf), HEAD_BATCH):
            b = perm[i:i + HEAD_BATCH]
            opt.zero_grad()
            loss = loss_fn(model(Xf[b]), yf[b])
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            logits = model(Xv)
            val_loss = float(loss_fn(logits, yv))
            val_acc = float((logits.argmax(1) == yv).float().mean())
        history.append({"epoch": epoch, "train_loss": round(total / len(Xf), 5),
                        "val_loss": round(val_loss, 5), "val_acc": round(val_acc, 4),
                        "elapsed_s": round(time.time() - t0, 2)})
        if val_loss < best["val_loss"] - 1e-4:
            best = {"val_loss": val_loss, "val_acc": val_acc, "epoch": epoch,
                    "state": copy.deepcopy(model.state_dict())}
            since_best = 0
        else:
            since_best += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f"    epoch {epoch:3d}  train loss {total / len(Xf):.4f}  val loss {val_loss:.4f}  "
                  f"val acc {val_acc:.4f}  elapsed {time.time() - t0:.1f}s")
        if since_best >= HEAD_PATIENCE:
            print(f"    early stop at epoch {epoch} (no val-loss improvement for {HEAD_PATIENCE} epochs); "
                  f"keeping epoch {best['epoch']}  elapsed {time.time() - t0:.1f}s")
            break
    model.load_state_dict(best["state"])
    model.eval()
    return model, best, history


def gather(mm, rows):
    # float32 copy of the given rows of a saved (float16) vector file, read in steps
    out = np.empty((len(rows), mm.shape[1]), dtype=np.float32)
    for a in range(0, len(rows), ROW_CHUNK):
        out[a:a + ROW_CHUNK] = mm[rows[a:a + ROW_CHUNK]]
    return out


def standardise(X_fit, *others):
    # StandardScaler fit on the fit rows (in steps, to keep memory low), applied in place to every array
    scaler = StandardScaler()
    for a in range(0, len(X_fit), ROW_CHUNK):
        scaler.partial_fit(X_fit[a:a + ROW_CHUNK])
    for X in (X_fit, *others):
        for a in range(0, len(X), ROW_CHUNK):
            X[a:a + ROW_CHUNK] = scaler.transform(X[a:a + ROW_CHUNK], copy=False)
    return scaler


def confusion(y_true_i, y_pred_i):
    # rows = true class, columns = predicted class (class positions 0..2)
    k = len(CLASSES)
    return np.bincount(y_true_i * k + y_pred_i, minlength=k * k).reshape(k, k)


def macro_f1_from(cm):
    tp = np.diag(cm).astype(float)
    denom = cm.sum(0) + cm.sum(1)               # 2*TP + FP + FN per class
    return float(np.mean(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)))


def macro_f1_many(cms):
    # macro-F1 of a stack of confusion matrices (B, k, k)
    tp = np.diagonal(cms, axis1=1, axis2=2)
    denom = cms.sum(1) + cms.sum(2)
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1e-12), 0.0).mean(1)


def evaluate(y_true, y_pred):
    # 03_train_head.py evaluate(): accuracy, macro-F1, per-class precision/recall/F1, confusion matrix
    from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                                 precision_recall_fscore_support)

    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=CLASSES, zero_division=0)
    return {
        "macro_f1": round(float(f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0)), 4),
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "per_class": {LABEL_NAMES[c]: {"label": int(c), "precision": round(float(p[i]), 4),
                                       "recall": round(float(r[i]), 4), "f1": round(float(f[i]), 4),
                                       "support": int(s[i])} for i, c in enumerate(CLASSES)},
        "confusion_matrix": {"rows": "true label", "cols": "predicted label", "names": CLASS_NAMES,
                             "matrix": confusion_matrix(y_true, y_pred, labels=CLASSES).tolist()},
    }


def interval(values):
    lo, hi = np.nanpercentile(values, [2.5, 97.5])
    return [round(float(lo), 4), round(float(hi), 4)]


# The same gene redraws for every model: GENE_COUNTS[b, g] = how often gene g is drawn in redraw b.
_draws = np.random.default_rng(SEED).integers(0, len(GENES), size=(N_BOOT, len(GENES)))
GENE_COUNTS = np.stack([np.bincount(r, minlength=len(GENES)) for r in _draws]).astype(np.float64)
del _draws


def gene_bootstrap(pred_i):
    # Every score on every gene redraw. A redraw's confusion matrix is the sum of the drawn genes'
    # matrices, so it is GENE_COUNTS @ (per-gene confusion matrices).
    k, G = len(CLASSES), len(GENES)
    per_gene = np.bincount(GENE_CODE * k * k + yi["test"] * k + pred_i, minlength=G * k * k).reshape(G, k * k)
    cms = (GENE_COUNTS @ per_gene).reshape(N_BOOT, k, k)
    boot = {"macro_f1": macro_f1_many(cms), "accuracy": np.trace(cms, axis1=1, axis2=2) / cms.sum((1, 2))}
    for g in NEG_GROUPS:
        m = TEST_GROUP == g
        n_g = np.bincount(GENE_CODE[m], minlength=G)
        hit = np.bincount(GENE_CODE[m], weights=(pred_i[m] == NONE_I).astype(float), minlength=G)
        tot = GENE_COUNTS @ n_g
        with np.errstate(invalid="ignore", divide="ignore"):
            boot[g] = np.where(tot > 0, (GENE_COUNTS @ hit) / tot, np.nan)
    return boot


def window_bootstrap(pred_i):
    # macro-F1 on redraws of single windows (seeded, so the same redraws for every model); for comparison only
    rng, n = np.random.default_rng(SEED), len(pred_i)
    f1 = np.empty(N_BOOT)
    for b in range(N_BOOT):
        r = rng.integers(0, n, n)
        f1[b] = macro_f1_from(confusion(yi["test"][r], pred_i[r]))
    return f1


def train_and_score(spec):
    # Standardise, train the head, score the test windows once.
    F = features(spec)
    X_fit, X_val = gather(F["train"], fit_rows), gather(F["train"], val_rows)
    X_test = gather(F["test"], np.arange(len(y["test"])))
    del F
    scaler = standardise(X_fit, X_val, X_test)
    t0 = time.time()
    head, best, hist = train_mlp(X_fit, yi["train"][fit_rows], X_val, yi["train"][val_rows])
    head_s = time.time() - t0
    in_dim = int(X_fit.shape[1])
    del X_fit, X_val
    gc.collect()
    with torch.no_grad():
        prob = torch.softmax(head(torch.from_numpy(X_test)), dim=1).numpy()
    del X_test
    pred_i = prob.argmax(1)
    pred = np.array(CLASSES)[pred_i]
    m = evaluate(y["test"], pred)
    assert abs(macro_f1_from(confusion(yi["test"], pred_i)) - m["macro_f1"]) < 1e-4    # my F1 == sklearn's F1
    boot = gene_bootstrap(pred_i)
    boot["macro_f1_windows"] = window_bootstrap(pred_i)
    m["macro_f1_ci95"], m["accuracy_ci95"] = interval(boot["macro_f1"]), interval(boot["accuracy"])
    m["macro_f1_ci95_window_resampling"] = interval(boot["macro_f1_windows"])
    m["negatives_called_none"] = {
        g: {"n": int((TEST_GROUP == g).sum()),
            "called_none": round(float((pred_i[TEST_GROUP == g] == NONE_I).mean()), 4) if (TEST_GROUP == g).any() else None,
            "ci95": interval(boot[g]) if (TEST_GROUP == g).any() else None} for g in NEG_GROUPS}
    training = {
        "features": "center_plus_prev_concat, rounded through float16, StandardScaler fit on the fit rows",
        "head": f"Linear({in_dim},256) -> ReLU -> Dropout(0.2) -> Linear(256,{len(CLASSES)})",
        "optimizer": f"Adam(lr={HEAD_LR}, weight_decay={HEAD_WEIGHT_DECAY})", "batch_size": HEAD_BATCH,
        "max_epochs": HEAD_MAX_EPOCHS, "patience": HEAD_PATIENCE, "selection": "lowest validation loss",
        "n_fit": int(len(fit_rows)), "n_val": int(len(val_rows)), "seed": HEAD_SEED,
        "best_epoch": best["epoch"], "epochs_run": len(hist),
        "best_val_loss": round(best["val_loss"], 5), "best_val_acc": round(best["val_acc"], 4),
        "train_seconds": round(head_s, 1), "history": hist,
    }
    np.savez(PRED_DIR / f"{spec['key']}.npz", test_row=idx["test"], y_true=y["test"], y_pred=pred,
             prob=prob.astype(np.float16), gene_id=TEST_GENE, group=TEST_GROUP)
    torch.save({"state_dict": head.state_dict(), "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_,
                "classes": CLASSES, "label_names": LABEL_NAMES, "in_dim": in_dim,
                "model": spec["repo"], "revision": spec["revision"]}, HEAD_DIR / f"{spec['key']}_head.pt")
    return m, training, boot


def print_report(title, m):
    print(f"  --- {title} ---")
    print(f"  test macro-F1 {m['macro_f1']:.4f}  (95% interval, genes resampled: {m['macro_f1_ci95'][0]:.4f}-"
          f"{m['macro_f1_ci95'][1]:.4f})   accuracy {m['accuracy']:.4f}")
    for name, v in m["per_class"].items():
        print(f"  {name:>10}  precision {v['precision']:.3f}  recall {v['recall']:.3f}  n {v['support']}")
    print("  negatives called none: " + "   ".join(
        f"{g} {v['called_none']:.3f} (n={v['n']})" for g, v in m["negatives_called_none"].items() if v["n"]))
    print("  confusion matrix (rows = true, columns = predicted): " + " / ".join(CLASS_NAMES))
    for name, row in zip(CLASS_NAMES, m["confusion_matrix"]["matrix"]):
        print(f"  {name:>10}  " + "  ".join(f"{v:6d}" for v in row))

## Figures

One **confusion matrix** per model shows, for each true class, where its windows ended up; a good model has nearly everything on the diagonal. The **comparison chart** puts the two models side by side on macro-F1, with the gene-level 95% interval as a line through each dot. The axis is zoomed in on the range where the scores are, so read the numbers, not the distances from the left edge. (Same style as `compare_models.ipynb`.)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

SURFACE, INK, INK_2, GRID, BLUE = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1", "#2a78d6"
BLUE_RAMP = LinearSegmentedColormap.from_list("blue_ramp", [SURFACE, "#cde2fb", "#86b6ef", "#3987e5",
                                                           "#1c5cab", "#0d366b"])
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "font.size": 10, "text.color": INK, "axes.labelcolor": INK_2, "axes.edgecolor": GRID,
                     "xtick.color": INK_2, "ytick.color": INK_2})
SMOKE_TAG = "SMOKE RUN, not a result.  " if SMOKE else ""


def show(fig, path):
    fig.savefig(path, dpi=160, bbox_inches="tight")
    if "inline" in matplotlib.get_backend().lower():
        plt.show()
    plt.close(fig)


def plot_confusion(spec, m):
    # Colour = share of the true class (row); each cell also shows the count.
    cm = np.array(m["confusion_matrix"]["matrix"])
    share = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(4.6, 4.3))
    ax.imshow(share, cmap=BLUE_RAMP, vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:,}\n{share[i, j]:.1%}", ha="center", va="center", fontsize=10,
                    color="white" if share[i, j] > 0.55 else INK)
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_xticks(np.arange(-0.5, cm.shape[1]), minor=True)
    ax.set_yticks(np.arange(-0.5, cm.shape[0]), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="minor", length=0)
    ax.set_title(f"{spec['name']}: genome test confusion matrix\n{SMOKE_TAG}macro-F1 {m['macro_f1']:.3f}, "
                 f"accuracy {m['accuracy']:.3f}, {int(cm.sum()):,} windows", fontsize=10, loc="left", color=INK)
    show(fig, FIG_DIR / f"confusion_{spec['key']}.png")


def plot_comparison(done):
    # One dot per model: test macro-F1 with its gene-level 95% interval, best at the top.
    done = sorted(done, key=lambda r: r["test"]["macro_f1"])
    fig, ax = plt.subplots(figsize=(8.6, 1.5 + 0.62 * len(done)))
    lo = min(r["test"]["macro_f1_ci95"][0] for r in done)
    hi = max(r["test"]["macro_f1_ci95"][1] for r in done)
    pad = max(0.005, 0.12 * (hi - lo))
    for k, r in enumerate(done):
        t, e = r["test"], r["embedding"]
        ax.plot(t["macro_f1_ci95"], [k, k], color=BLUE, linewidth=2, solid_capstyle="round")
        ax.plot([t["macro_f1"]], [k], "o", color=BLUE, markersize=9, markeredgecolor=SURFACE, markeredgewidth=2)
        ax.text(t["macro_f1"], k + 0.24, f"{t['macro_f1']:.3f}", ha="center", va="bottom", fontsize=10, color=INK)
        cost = f"accuracy {t['accuracy']:.3f}   embedding {e.get('embed_seconds', float('nan')) / 60:.1f} min"
        if e.get("peak_gpu_gb") is not None:
            cost += f"   peak GPU {e['peak_gpu_gb']:.1f} GB"
        ax.text(1.02, k, cost, transform=ax.get_yaxis_transform(), ha="left", va="center", fontsize=9, color=INK_2)
    ax.set_yticks(range(len(done)), [f"{r['name']}\n{r['embedding'].get('parameters', 0) / 1e6:,.0f}M parameters"
                                     for r in done])
    ax.set_ylim(-0.6, len(done) - 0.3)
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_xlabel("test macro-F1 (dot) with 95% interval from resampling genes (line); axis does not start at 0")
    ax.grid(axis="x", color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.set_title(f"{SMOKE_TAG}Frozen models on the genome dataset: same head, {len(fit_rows) + len(val_rows):,} "
                 f"training windows,\n{len(y['test']):,} test windows from {len(GENES)} genes (chr20/21)",
                 fontsize=11, loc="left", color=INK)
    show(fig, FIG_DIR / "comparison.png")

## Run both models

This is the cell that takes the time. For each model, in order: find and check the centre token in every window, embed every window (with checkpoints), train the head, score it on the test windows, draw its confusion matrix, and update `results.json`. Because `results.json` is written after **each** model and the vectors after every chunk, a crash loses very little.

In [ ]:
results = {
    "smoke": SMOKE,
    "note": ("SMOKE run: tiny sample, one model. These numbers only show the notebook runs."
             if SMOKE else "Full run: the 120,000-window default train file and every chr20/21 test window."),
    "task": "3-class splice-site windows (none / acceptor / donor), frozen backbone + small MLP head",
    "dataset": "genome-wide splice-site windows (scripts/16_genome_dataset.py), Kaggle package of "
               "scripts/19_kaggle_genome_package.py",
    "data_files": {f: DATA_SHA[s] for s, f in DATA_FILES.items()},
    "split": {"train_chromosomes": sorted(chroms["train"], key=lambda c: (len(c), c)), "test_chromosomes": "chr20, chr21",
              "n_train": int(len(y["train"])), "n_fit": int(len(fit_rows)), "n_val": int(len(val_rows)),
              "n_test": int(len(y["test"])), "test_genes": int(len(GENES)),
              "test_kinds": {g: int((TEST_GROUP == g).sum()) for g in ["donor", "acceptor"] + NEG_GROUPS},
              "val_split_seed": HEAD_SEED, "rows": ROWS_KEY},
    "features": "last-layer vector of the token covering base 300, concatenated with the token before it",
    "device": str(DEVICE), "gpu": GPU_NAME, "gpu_compute_capability": GPU_CC,
    "gpu_total_gb": round(GPU_TOTAL_GB, 1) if ON_GPU else None,
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "sklearn": sklearn.__version__},
    "reference_model": REFERENCE, "challenger_model": CHALLENGER,
    "bootstrap": {"resamples": N_BOOT, "unit": "test genes (gene_id), drawn with replacement; "
                  "the same redraws for every model", "seed": SEED},
    "run_order": [m["key"] for m in RUN_MODELS], "models": {},
}
BOOT = {}
for n_model, spec in enumerate(RUN_MODELS, start=1):
    t_model = time.time()
    print(f"\n===== {n_model}/{len(RUN_MODELS)}  {spec['name']}  ({spec['repo']}) =====")
    entry = {"name": spec["name"], "repo": spec["repo"], "revision": spec["revision"]}
    try:
        emb_info, centre = embed_model(spec)
        print(f"  training the head on {len(fit_rows):,} windows, validating on {len(val_rows):,}")
        metrics, training, BOOT[spec["key"]] = train_and_score(spec)
        entry.update({"status": "ok", "embedding": emb_info, "centre_token": centre,
                      "head_training": training, "test": metrics})
        print_report(f"{spec['name']}: frozen backbone + head (n={len(y['test']):,})", metrics)
        plot_confusion(spec, metrics)
    except Exception as exc:
        entry.update({"status": "skipped", "reason": f"{type(exc).__name__}: {str(exc)[:1500]}"})
        print(f"  SKIPPED {spec['name']}: {entry['reason'][:600]}")
    free_gpu()
    entry["seconds_total"] = round(time.time() - t_model, 1)
    results["models"][spec["key"]] = entry
    results["total_seconds"] = round(time.time() - T_START, 1)
    write_json_atomic(RESULTS_JSON, results)
    print(f"  {spec['name']} done in {entry['seconds_total'] / 60:.1f} min; results.json updated "
          f"({(time.time() - T_START) / 60:.1f} min since the start)")

## Compare, write the summary, and pack everything into one zip

The table ranks the models by test macro-F1. The difference row is NT 2.5B minus NT-v2 50M, on the same gene redraws: if its interval does not include 0, the gap is unlikely to be luck. The same is done for the share of GT and AG decoys called none.

Everything goes into **`/kaggle/working/genome_nt2500m.zip`**: `results.json` (every number, including each head's per-epoch training history), `summary.md` (the tables, ready for the repo), the figures, each model's test predictions (`predictions/<model>.npz`: test row, true label, predicted label, class probabilities, gene and kind of window), the trained heads, and, with `ZIP_EMBEDDINGS = True`, the saved vectors (about 1.7 GB). With those, other heads can be trained later on the laptop, with no GPU.

In [ ]:
done = [dict(results["models"][m["key"]], key=m["key"]) for m in RUN_MODELS
        if results["models"][m["key"]]["status"] == "ok"]
skipped = [results["models"][m["key"]] for m in RUN_MODELS if results["models"][m["key"]]["status"] != "ok"]
ok = {r["key"]: r for r in done}

# CHALLENGER minus REFERENCE, on the same gene redraws (and, for comparison, the same window redraws)
DIFF = None
if REFERENCE in BOOT and CHALLENGER in BOOT:
    ref_t, ch_t = ok[REFERENCE]["test"], ok[CHALLENGER]["test"]
    DIFF = {"challenger": CHALLENGER, "reference": REFERENCE,
            "macro_f1": {"difference": round(ch_t["macro_f1"] - ref_t["macro_f1"], 4),
                         "ci95": interval(BOOT[CHALLENGER]["macro_f1"] - BOOT[REFERENCE]["macro_f1"]),
                         "ci95_window_resampling": interval(BOOT[CHALLENGER]["macro_f1_windows"]
                                                            - BOOT[REFERENCE]["macro_f1_windows"])},
            "accuracy": {"difference": round(ch_t["accuracy"] - ref_t["accuracy"], 4),
                         "ci95": interval(BOOT[CHALLENGER]["accuracy"] - BOOT[REFERENCE]["accuracy"])}}
    for g in NEG_GROUPS:
        a, b = ch_t["negatives_called_none"][g], ref_t["negatives_called_none"][g]
        if a["n"]:
            DIFF[f"{g}_called_none"] = {"difference": round(a["called_none"] - b["called_none"], 4),
                                        "ci95": interval(BOOT[CHALLENGER][g] - BOOT[REFERENCE][g])}
results["challenger_minus_reference"] = DIFF
if REFERENCE in ok and not SMOKE:
    t = ok[REFERENCE]["test"]
    results["reference_vs_laptop"] = {
        "laptop": LAPTOP_50M, "source": "results/learning_curve/summary.md (120,000 windows, mean of 3 seeds)",
        "here": {"macro_f1": t["macro_f1"], "decoy_GT": t["negatives_called_none"]["decoy_GT"]["called_none"],
                 "decoy_AG": t["negatives_called_none"]["decoy_AG"]["called_none"]}}
results["total_seconds"] = round(time.time() - T_START, 1)
write_json_atomic(RESULTS_JSON, results)

# ---------- printed table ----------
if SMOKE:
    print("SMOKE RUN: tiny sample. Not a result. Set SMOKE = False for the real numbers.\n")
print(f"{'model':<12}{'macro-F1':>9}{'95% CI (genes)':>18}{'accuracy':>10}"
      + "".join(f"{'recall ' + n:>16}" for n in CLASS_NAMES) + f"{'GT decoys':>11}{'AG decoys':>11}{'embed min':>11}")
for r in sorted(done, key=lambda r: -r["test"]["macro_f1"]):
    t, e, neg = r["test"], r["embedding"], r["test"]["negatives_called_none"]
    print(f"{r['name']:<12}{t['macro_f1']:>9.4f}{t['macro_f1_ci95'][0]:>10.4f}-{t['macro_f1_ci95'][1]:.4f}"
          f"{t['accuracy']:>10.4f}" + "".join(f"{t['per_class'][n]['recall']:>16.4f}" for n in CLASS_NAMES)
          + "".join(f"{neg[g]['called_none']:>11.3f}" if neg[g]["n"] else f"{'-':>11}" for g in ("decoy_GT", "decoy_AG"))
          + f"{e.get('embed_seconds', float('nan')) / 60:>11.1f}")
for r in skipped:
    print(f"{r['name']:<12}SKIPPED: {r['reason'][:200]}")
if DIFF:
    d = DIFF["macro_f1"]
    print(f"\n{SPECS[CHALLENGER]['name']} minus {SPECS[REFERENCE]['name']}: macro-F1 {d['difference']:+.4f}, "
          f"95% CI {d['ci95'][0]:+.4f} to {d['ci95'][1]:+.4f} (genes resampled; "
          f"windows resampled: {d['ci95_window_resampling'][0]:+.4f} to {d['ci95_window_resampling'][1]:+.4f})")
    for g in ("decoy_GT", "decoy_AG"):
        if f"{g}_called_none" in DIFF:
            v = DIFF[f"{g}_called_none"]
            print(f"  {g} called none: {v['difference']:+.4f}, 95% CI {v['ci95'][0]:+.4f} to {v['ci95'][1]:+.4f}")
if "reference_vs_laptop" in results:
    h = results["reference_vs_laptop"]["here"]
    print(f"\nNT-v2 50M here vs on the laptop (3-seed mean): macro-F1 {h['macro_f1']:.4f} vs {LAPTOP_50M['macro_f1']}, "
          f"GT decoys {h['decoy_GT']:.3f} vs {LAPTOP_50M['decoy_GT']}, AG decoys {h['decoy_AG']:.3f} vs {LAPTOP_50M['decoy_AG']}")
if done:
    plot_comparison(done)


# ---------- summary.md ----------
def ci(v, sign=False, d=4):
    f = "{:+.%df}" % d if sign else "{:.%df}" % d
    return f"[{f.format(v[0])}, {f.format(v[1])}]" if v else ""


order = sorted(done, key=lambda r: -r["test"]["macro_f1"])
sp = results["split"]
L = [f"# {SMOKE_TAG}NT 2.5B vs NT-v2 50M on the genome-wide splice-site dataset", "",
     f"_Numbers from `results.json`, written by `kaggle/genome_nt2500m.ipynb` on {GPU_NAME or 'CPU'} "
     f"({results['total_seconds'] / 60:.0f} min in total)._", "",
     f"Frozen models, the same features (centre token + previous token) and the same 03 head for both. "
     f"Train: {sp['n_train']:,} windows (every chromosome except chr20/21; {sp['n_fit']:,} to fit, {sp['n_val']:,} validation). "
     f"Test: {sp['n_test']:,} windows on chr20/21 from {sp['test_genes']} genes. 95% intervals: "
     f"{N_BOOT:,} redraws of the test genes, the same redraws for both models.", "",
     "| model | macro-F1 | 95% CI (genes) | accuracy | difference vs NT-v2 50M | 95% CI of the difference |",
     "|---|---:|---|---:|---:|---|"]
for r in order:
    t = r["test"]
    if r["key"] == REFERENCE:
        diff = ("reference", "")
    elif DIFF and r["key"] == CHALLENGER:
        diff = (f"{DIFF['macro_f1']['difference']:+.4f}", ci(DIFF["macro_f1"]["ci95"], True))
    else:
        diff = ("", "")
    L.append(f"| {r['name']} | {t['macro_f1']:.4f} | {ci(t['macro_f1_ci95'])} | {t['accuracy']:.4f} | {diff[0]} | {diff[1]} |")
if DIFF:
    L += ["", f"Window-level intervals for comparison: the difference's interval would be "
              f"{ci(DIFF['macro_f1']['ci95_window_resampling'], True)} if single windows were resampled."]
L += ["", "## Negatives called none (recall on each kind of negative)", "",
      "| model | " + " | ".join(NEG_GROUPS) + " |", "|---|" + "---|" * len(NEG_GROUPS)]
for r in order:
    neg = r["test"]["negatives_called_none"]
    L.append(f"| {r['name']} | " + " | ".join(
        f"{neg[g]['called_none']:.3f} {ci(neg[g]['ci95'], d=3)}" if neg[g]["n"] else "-" for g in NEG_GROUPS) + " |")
if DIFF:
    L.append("| difference | " + " | ".join(
        f"{DIFF[g + '_called_none']['difference']:+.3f} {ci(DIFF[g + '_called_none']['ci95'], True, d=3)}"
        if g + "_called_none" in DIFF else "-" for g in NEG_GROUPS) + " |")
L += ["", "n = " + ", ".join(f"{g} {sp['test_kinds'][g]:,}" for g in NEG_GROUPS) + ".", "",
      "## Per class (test)", "",
      "| model | " + " | ".join(f"{n} precision | {n} recall" for n in CLASS_NAMES) + " |",
      "|---|" + "---:|" * (2 * len(CLASS_NAMES))]
for r in order:
    pc = r["test"]["per_class"]
    L.append(f"| {r['name']} | " + " | ".join(f"{pc[n]['precision']:.3f} | {pc[n]['recall']:.3f}" for n in CLASS_NAMES) + " |")
L += ["", "## Confusion matrices (test; rows = true label, columns = predicted)", ""]
for r in order:
    cm = r["test"]["confusion_matrix"]["matrix"]
    L += [f"**{r['name']}**", "", "| true \\ predicted | " + " | ".join(CLASS_NAMES) + " |",
          "|---|" + "---:|" * len(CLASS_NAMES)]
    L += [f"| {n} | " + " | ".join(f"{v:,}" for v in row) + " |" for n, row in zip(CLASS_NAMES, cm)]
    L += ["", f"![{r['name']} confusion matrix](figures/confusion_{r['key']}.png)", ""]
L += ["![comparison](figures/comparison.png)", "", "## Run details", "",
      "| model | precision on the GPU | embedding | peak GPU memory | head best epoch / epochs run |",
      "|---|---|---:|---:|---:|"]
for r in done:
    e, h = r["embedding"], r["head_training"]
    L.append(f"| {r['name']} | {e.get('precision', '-')} | {e.get('embed_seconds', float('nan')) / 60:.1f} min | "
             + (f"{e['peak_gpu_gb']:.1f} GB" if e.get("peak_gpu_gb") is not None else "-")
             + f" | {h['best_epoch']} / {h['epochs_run']} |")
for r in skipped:
    L.append(f"| {r['name']} | skipped: {r['reason'][:150]} | | | |")
v = results["versions"]
L += ["", f"GPU: {GPU_NAME or 'none'}. Versions: torch {v['torch']}, transformers {v['transformers']}, "
      f"scikit-learn {v['sklearn']}. Model revisions and data-file hashes are in `results.json`.", ""]
SUMMARY_MD.write_text("\n".join(L))
print(f"\nwrote {SUMMARY_MD.name}")

# ---------- one zip ----------
files = [RESULTS_JSON, SUMMARY_MD] + sorted(FIG_DIR.glob("*.png")) + sorted(PRED_DIR.glob("*.npz")) + sorted(HEAD_DIR.glob("*.pt"))
emb_files = sorted(f for f in EMB_DIR.rglob("*") if f.is_file() and f.suffix in (".npy", ".json"))
emb_gb = sum(f.stat().st_size for f in emb_files) / 1e9
with_emb = ZIP_EMBEDDINGS and free_disk_gb(BASE_DIR) > emb_gb + 1.0
if ZIP_EMBEDDINGS and not with_emb:
    print(f"not enough disk to also zip the vectors ({emb_gb:.1f} GB): zipping without them; "
          f"they stay in {EMB_DIR} in the notebook's output")
tmp = ZIP_PATH.with_suffix(".zip.tmp")
with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as z:
    for f in files:
        z.write(f, Path(OUT_DIR.name) / f.relative_to(OUT_DIR))
    for f in (emb_files if with_emb else []):
        z.write(f, Path(OUT_DIR.name) / f.relative_to(OUT_DIR), compress_type=zipfile.ZIP_STORED)
os.replace(tmp, ZIP_PATH)
print(f"wrote {ZIP_PATH}: {len(files) + (len(emb_files) if with_emb else 0)} files, "
      f"{ZIP_PATH.stat().st_size / 1e6:,.1f} MB" + (" (vectors included)" if with_emb else ""))
print(f"\n{len(done)} of {len(RUN_MODELS)} models finished, {len(skipped)} skipped.   device: {DEVICE}   "
      f"total time: {(time.time() - T_START) / 60:.1f} min")